# Snowflake → AIDP migration: environment diagnosis

Run this INSIDE AIDP (any cluster) before the migration jobs. It answers the
four questions that cost a day to answer the hard way, in order, and each
check prints a verdict rather than a traceback:

1. **Where do workspace files land on this cluster?** (`/Workspace` on the
   validated build.)
2. **Can this cluster reach Snowflake at all?** (TCP 443 to the account host.)
3. **Do the credentials work through the AIDP Snowflake connector?**
   (`current_user()` via pushdown — the path the migration scripts use.)
4. **Does the registered EXTERNAL catalog actually expose anything?**
   (Its crawler can fail while the connector works — observed live.)

Fill in `CONFIG` below, or point `CONFIG_PATH` at the JSON connection config
already on the workspace. Nothing here writes anything anywhere.

In [ ]:
# --- parameters ----------------------------------------------------------
CONFIG_PATH = "/Workspace/backup-snowflake-migration/plan/snowmig_source_config.json"
EXTERNAL_CATALOG = ""      # optional: the registered EXTERNAL catalog name
SESSION_SCHEMA = ""        # a REAL schema; required for the pushdown check
SCRIPTS_DIR = "/Workspace/backup-snowflake-migration/scripts"

In [ ]:
# --- 1. the workspace mount ----------------------------------------------
import json, os, socket, sys

def verdict(name, ok, detail=""):
    print(f"[{'PASS' if ok else 'FAIL'}] {name}" + (f" — {detail}" if detail else ""), flush=True)
    return ok

found = [p for p in ("/Workspace", "/workspace", "/mnt/workspace") if os.path.isdir(p)]
verdict("workspace mount", bool(found), f"visible at {found}" if found else
        "no workspace tree on this filesystem — a job cannot read uploaded files")
if os.path.isdir(SCRIPTS_DIR):
    verdict("migration scripts present", True, f"{sorted(os.listdir(SCRIPTS_DIR))[:6]}")
else:
    verdict("migration scripts present", False, f"{SCRIPTS_DIR} not found")

In [ ]:
# --- 2. the config, read back and echoed (never the secret) --------------
config = {}
try:
    with open(CONFIG_PATH) as fh:
        config = json.load(fh)
    shown = {k: ("<path>" if "key" in k or "password" in k else v)
             for k, v in config.items()}
    verdict("source config readable", True, json.dumps(shown))
except Exception as exc:
    verdict("source config readable", False, f"{type(exc).__name__}: {exc}")

host = config.get("host") or (f"{config.get('account','')}.snowflakecomputing.com"
                              if config.get("account") else "")
print("   host to be used:", host or "(unknown)")
print("   CONFIRM WITH THE USER: account, user, role, warehouse, database above.")

In [ ]:
# --- 3. network egress from THIS cluster ---------------------------------
if host:
    try:
        socket.create_connection((host, 443), timeout=10).close()
        verdict("cluster → Snowflake :443", True, host)
    except Exception as exc:
        verdict("cluster → Snowflake :443", False,
                f"{type(exc).__name__}: {exc} — the cluster has no route; the "
                f"connector path cannot work either")

In [ ]:
# --- 4. the AIDP Snowflake connector, with these credentials -------------
sys.path.insert(0, SCRIPTS_DIR)
try:
    from snowmig_source import SnowflakeSource
    source = SnowflakeSource(spark, mode="connector", config=config,
                             session_schema=SESSION_SCHEMA or None)
    rows = source.pushdown(
        "select current_user() U, current_role() R, current_warehouse() W, "
        "count(*) N from INFORMATION_SCHEMA.TABLES").collect()
    verdict("connector pushdown", True, str([r.asDict() for r in rows]))
except Exception as exc:
    verdict("connector pushdown", False, f"{type(exc).__name__}: {str(exc)[:400]}")
    print("   DATA_ACCESS_LAYER_0031 here means SESSION_SCHEMA is not a real "
          "schema; the connector refuses INFORMATION_SCHEMA in that option.")

In [ ]:
# --- 5. the EXTERNAL catalog: has its crawler actually populated it? -----
if EXTERNAL_CATALOG:
    try:
        schemas = spark.sql(f"SHOW SCHEMAS IN `{EXTERNAL_CATALOG}`").collect()
        verdict("external catalog populated", bool(schemas),
                f"{len(schemas)} schema(s)" if schemas else
                "registered but EMPTY — the crawler has not succeeded. Check "
                "its refresh status; a crawl can fail (CONNECTOR_0067, "
                "'Login has timed out') while the connector above works, "
                "because the crawler runs outside the cluster's network path")
    except Exception as exc:
        verdict("external catalog populated", False, f"{type(exc).__name__}: {str(exc)[:300]}")
else:
    print("[SKIP] external catalog — EXTERNAL_CATALOG not set. A skip is not a pass.")

## Reading the result

| Pattern | What it means | What to do |
|---|---|---|
| connector PASS, catalog EMPTY | the credentials are fine; the crawler cannot reach Snowflake | run the migration with `--source-mode connector` (the default) |
| egress FAIL | the cluster has no route to Snowflake | fix the cluster's network/NAT before anything else |
| connector FAIL, egress PASS | credentials, role, warehouse or key | re-confirm the config fields with the user, then retry |
| mount FAIL | jobs cannot read uploaded files | do not run the jobs; re-check the workspace upload |